# Pre-process the MCES dataset

The MCES dataset (Kretschmer et al, 2025) consists of an .npz file containing two fields: smiles and mces. The former contains smiles strings while the latter contains pairwise weighted maximum common edge substructure distance computations. The MCES calculations are stored in upper-triangle row-major order.

As the notebook will show, many of the molecules in the MCES dataset are not fit for use to compare molecular representations. This notebook performs the pre-processing needed to clean the dataset before use. We will create a 'censored' array, informing us which pairwise comparisons to skip when looking to quantify the relationship between Tanimoto/cosine similarity and global graph topology as measured by the MCES distance.

# Imports

In [18]:
import numpy as np
import rdkit
from rdkit import Chem
from collections import Counter
import pandas as pd
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.warning")
from rdkit.Chem import Descriptors

## Load the MCES dataset

In [ ]:
MOLBERT_VALID = 'path/to/MolBERT valid.npy'
COATI_INVALID = 'path/to/COATI invalid.npy'

In [ ]:
distances = np.load('/path/to/mces_distances.npz')
mces      = distances['mces']
smiles    = distances['smiles']

In [20]:
print(mces.shape)
print(smiles.shape)

(2012094766,)
(63437,)


## Check all valid smiles

In [21]:
invalid = 0
for s in smiles:
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        invalid += 1

if invalid == 0:
    print('All smiles valid')
else:
    print(f"Total invalid smiles: {invalid}")

All smiles valid


## Check all unique

In [22]:
unique = set()
for s in smiles:
    mol = Chem.MolFromSmiles(s)
    s   = Chem.MolToSmiles(mol, canonical=True)
    unique.add(s)

if len(unique) == len(smiles):
    print("All smiles unique")
else:
    print("At least one non-unique smiles")

All smiles unique


## Censor smiles strings

We will use a binary array to create a mask for censored smiles strings.

In [23]:
censored  = np.zeros(shape=len(smiles))

### Censor fragmented molecules

Approximately 2,000 molecules (~3%) are fragmented. We will drop those containing fragments. At least one embedding model cannot process these.

In [24]:
def fragment_stats(smiles_list):
    n_total      = len(smiles_list)
    n_fragmented = 0
    n_invalid    = 0

    smallest_fragment_heavy_atoms = []
    fragmented_examples           = []

    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)

        if mol is None:
            n_invalid += 1
            continue

        # Split disconnected components
        fragments = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)

        if len(fragments) > 1:
            n_fragmented += 1

            heavy_atom_counts = [
                frag.GetNumHeavyAtoms()
                for frag in fragments
            ]

            smallest = min(heavy_atom_counts)
            smallest_fragment_heavy_atoms.append(smallest)

            fragmented_examples.append({
                "index": i,
                "smiles": smi,
                "n_fragments": len(fragments),
                "fragment_heavy_atoms": heavy_atom_counts,
                "smallest_fragment_heavy_atoms": smallest,
            })

    distribution = (
        pd.Series(smallest_fragment_heavy_atoms)
        .value_counts()
        .sort_index()
        .rename_axis("smallest_fragment_heavy_atoms")
        .reset_index(name="count")
    )

    print(f"Total SMILES: {n_total}")
    print(f"Invalid SMILES: {n_invalid}")
    print(f"Fragmented SMILES: {n_fragmented}")

    if n_total > 0:
        print(
            f"Fragmented fraction: "
            f"{n_fragmented / n_total:.2%}"
        )

    return {
        "n_total": n_total,
        "n_invalid": n_invalid,
        "n_fragmented": n_fragmented,
        "smallest_fragment_heavy_atoms": smallest_fragment_heavy_atoms,
        "distribution": distribution,
        "fragmented_examples": pd.DataFrame(fragmented_examples),
    }

In [25]:
stats = fragment_stats(smiles)
stats["distribution"]

Total SMILES: 63437
Invalid SMILES: 0
Fragmented SMILES: 2052
Fragmented fraction: 3.23%


,smallest_fragment_heavy_atoms,count
0,0,34
1,1,1476
2,2,9
3,3,10
4,4,42
5,5,140
6,6,22
7,7,36
8,8,95
9,9,15


In [26]:
stats["fragmented_examples"].head(20)

,index,smiles,n_fragments,fragment_heavy_atoms,smallest_fragment_heavy_atoms
0,0,[2H]C1(C([N+]2=CC=CC=C2C3=CC=CC=[N+]31)([2H])[...,3,"[14, 1, 1]",1
1,1,[2H]C([2H])([2H])[C@@H](CC1=CC=CC=C1)N.[2H]C([...,2,"[10, 10]",10
2,2,[2H]C([2H])([2H])[C@@H](CC1=CC=CC=C1)N.[2H]C([...,3,"[10, 10, 5]",5
3,77,B.C1COCCN1,2,"[1, 6]",1
4,92,[Be+2].O.O.O.O.[O-]S(=O)(=O)[O-],6,"[1, 1, 1, 1, 1, 5]",1
5,95,[B-](F)(F)(F)F.CCCCCCCCCCCCCCCCN1C=C[N+](=C1)C,2,"[5, 22]",5
6,96,[B-](F)(F)(F)F.CCCCCCCCCCCCN1C=C[N+](=C1)C,2,"[5, 18]",5
7,97,[B-](F)(F)(F)F.CCCCCCCCCCN1C=C[N+](=C1)C,2,"[5, 16]",5
8,98,[B-](F)(F)(F)F.CCCCCCCCN1C=C[N+](=C1)C,2,"[5, 14]",5
9,99,[B-](F)(F)(F)F.CCCCCC[N+]1=CC=CC=C1,2,"[5, 12]",5


In [27]:
# Censor fragmented molecules
to_censor_fragments = stats["fragmented_examples"]['index'].to_list()
#to_censor_fragments = np.array(to_censor_fragments, dtype=bool)
censored[to_censor_fragments] = 1
print(censored.sum())

2052.0


### Censor molecules with rare atom types

In [28]:
permitted          = {'C', 'Cl', 'O', 'Br', 'I', 'F', 'N', 'S'}
to_censor_atoms    = []
for i, s in enumerate(smiles):
    mol        = Chem.MolFromSmiles(s)
    atom_types = set([atom.GetSymbol() for atom in mol.GetAtoms()])
    if not len((atom_types - permitted)) == 0:
        to_censor_atoms.append(1)
    else:
        to_censor_atoms.append(0)

to_censor_atoms           = np.array(to_censor_atoms, dtype=bool)
censored[to_censor_atoms] = 1

print(censored.sum())
to_censor_atoms.sum()

4869.0


np.int64(3517)

### Censor molecules with a molecular weight $>=$ 900

In [29]:
to_censor_molwt    = []
for i, s in enumerate(smiles):
    mol        = Chem.MolFromSmiles(s)
    if not Descriptors.MolWt(mol) < 900:
        to_censor_molwt.append(1)
    else:
        to_censor_molwt.append(0)

to_censor_molwt           = np.array(to_censor_molwt, dtype=bool)
censored[to_censor_molwt] = 1

print(censored.sum())
to_censor_molwt.sum()

6624.0


np.int64(2803)

### Censor molecules with no carbon atoms

In [30]:
to_censor_no_c = []
for i, s in enumerate(smiles):
    mol        = Chem.MolFromSmiles(s)
    atom_types = set([atom.GetSymbol() for atom in mol.GetAtoms()])
    if not 'C' in atom_types:
        to_censor_no_c.append(1)
    else:
        to_censor_no_c.append(0)

to_censor_no_c  = np.array(to_censor_no_c, dtype=bool)
censored[to_censor_no_c] = 1

print(censored.sum())
to_censor_no_c.sum()

6653.0


np.int64(146)

### Censor molecules that fail to be embedded

In [ ]:
molbert_valid   = np.load(MOLBERT_VALID)
molbert_invalid = ~molbert_valid
print(molbert_invalid.sum())

censored[molbert_invalid] = 1
print(censored.sum())

2528
6839.0


In [ ]:
coati_invalid   = np.load(COATI_INVALID)

print(coati_invalid.sum())

censored[coati_invalid] = 1
print(censored.sum())

2059
6839.0


### Create censored array

Molecules marked as censored are not to be compared against when computing 

In [33]:
def pairwise_censor_mask(censored):
    censored = np.asarray(censored, dtype=bool)
    n = censored.size

    m = n * (n - 1) // 2
    out = np.empty(m, dtype=bool)

    pos = 0
    for i in range(n - 1):
        width = n - i - 1

        if censored[i]:
            out[pos:pos + width] = True
        else:
            out[pos:pos + width] = censored[i + 1:]

        pos += width

    return out

In [34]:
censor_mask = pairwise_censor_mask(censored)
print(censor_mask.sum())
print(censor_mask.shape)

410456263
(2012094766,)


In [35]:
np.save('censor_mask.npy', censor_mask)